In [4]:
import pandas as pd

In [5]:
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [6]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [7]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [8]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [9]:
train_data.shape

(4000, 3)

# Data pre - processing

In [39]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.strip().lower()
    return text

In [40]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenizer

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [46]:
# raw - data -> tokenized inputs for fine tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [15]:
len(train_dataset[0]["input_ids"])

512

# Model

In [16]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 8922.89it/s]


In [17]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [18]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
# Training Arguments - Fine Tunning
training_args = TrainingArguments(
    output_dir = "./results", # saving best results

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8, # defaults also 8
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch", # when do we want to validate - that is after each epoch
    save_strategy = "epoch",

    warmup_steps = 500 # 1st slow training -> then fast
)

In [20]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.594451,0.700392
2,0.711764,0.628054
3,0.657320,0.603498
4,0.635692,0.592125
5,0.622231,0.583113
6,0.616104,0.581757


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.20it/s]


TrainOutput(global_step=3000, training_loss=1.1395935567220052, metrics={'train_runtime': 1253.3465, 'train_samples_per_second': 19.149, 'train_steps_per_second': 2.394, 'total_flos': 3248203235328000.0, 'train_loss': 1.1395935567220052, 'epoch': 6.0})

In [42]:
# Model Save
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.48it/s]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [43]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 9134.42it/s]


# Test the core logic of summarization

In [44]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    ).to(device)

    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )
    summary = tokenizer.decode(targets[0], skip_special_tokens = True)
    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform increasingly complex tasks.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency remains important.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand their decisions.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be essential to ensure AI develops in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print("Summary- ", summary)

Summary -  ai technology continues to expand rapidly across industries, from healthcare to finance. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
